## Data and Libraries

##### Imports

In [13]:
pip install requests


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [14]:
# Import libraries
import pandas as pd
import requests
import time
import calendar
from pathlib import Path



In [15]:
# Create a folder to save our project data
DATA_DIR = Path("malaria_project_data")
DATA_DIR.mkdir(exist_ok=True)

# Set our study period
START_YEAR = 2000
END_YEAR = 2023

##### Pull Sub-Saharan African country list from World Bank

In [53]:
url = "https://api.worldbank.org/v2/country"

params = {
    "format": "json",
    "per_page": 400
}

response = requests.get(url, params=params)
json_data = response.json()

countries = []

if isinstance(json_data, list) and len(json_data) > 1:
    country_list = json_data[1]
    
    for item in country_list:
        if item.get("region") and item.get("incomeLevel"):
            # 🟢 Added .strip() to clean up hidden trailing spaces from the API
            region = item["region"]["value"].strip()
            income_level = item["incomeLevel"]["value"].strip()
            
            if region == "Sub-Saharan Africa" and income_level != "Aggregates":
                countries.append({
                    "country_code": item["id"],
                    "country_name": item["name"],
                    "capital_city": item.get("capitalCity", ""),
                    "longitude": item.get("longitude", ""),
                    "latitude": item.get("latitude", "")
                })

# Create the final DataFrame
countries_df = pd.DataFrame(countries)
print(f"✅ Success! Loaded {len(countries_df)} countries.")


✅ Success! Loaded 48 countries.


In [54]:
countries_df.head()

,country_code,country_name,capital_city,longitude,latitude
0,AGO,Angola,Luanda,13.242,-8.81155
1,BDI,Burundi,Bujumbura,29.3639,-3.3784
2,BEN,Benin,Porto-Novo,2.6323,6.4779
3,BFA,Burkina Faso,Ouagadougou,-1.53395,12.3605
4,BWA,Botswana,Gaborone,25.9201,-24.6544


In [55]:
print(countries_df.columns)
print(countries_df.head(2))


Index(['country_code', 'country_name', 'capital_city', 'longitude',
       'latitude'],
      dtype='str')
  country_code country_name capital_city longitude  latitude
0          AGO       Angola       Luanda    13.242  -8.81155
1          BDI      Burundi    Bujumbura   29.3639   -3.3784


In [56]:
# Convert longitude and latitude to numeric values
countries_df["longitude"] = pd.to_numeric(countries_df["longitude"], errors="coerce")
countries_df["latitude"] = pd.to_numeric(countries_df["latitude"], errors="coerce")
    

In [57]:
countries_df.head(20)

,country_code,country_name,capital_city,longitude,latitude
0,AGO,Angola,Luanda,13.24200,-8.81155
1,BDI,Burundi,Bujumbura,29.36390,-3.37840
2,BEN,Benin,Porto-Novo,2.63230,6.47790
3,BFA,Burkina Faso,Ouagadougou,-1.53395,12.36050
4,BWA,Botswana,Gaborone,25.92010,-24.65440
5,CAF,Central African Republic,Bangui,21.64070,5.63056
6,CIV,Cote d'Ivoire,Yamoussoukro,-4.03050,5.33200
7,CMR,Cameroon,Yaounde,11.51740,3.87210
8,COD,"Congo, Dem. Rep.",Kinshasa,15.32220,-4.32500
9,COG,"Congo, Rep.",Brazzaville,15.26620,-4.27670


In [59]:
# 🟢 Remove countries without coordinates
countries_df = countries_df.dropna(subset=["longitude", "latitude"])

# Print results
print(countries_df.head())
print("Number of countries:", countries_df.shape[0])


  country_code  country_name capital_city  longitude  latitude
0          AGO        Angola       Luanda   13.24200  -8.81155
1          BDI       Burundi    Bujumbura   29.36390  -3.37840
2          BEN         Benin   Porto-Novo    2.63230   6.47790
3          BFA  Burkina Faso  Ouagadougou   -1.53395  12.36050
4          BWA      Botswana     Gaborone   25.92010 -24.65440
Number of countries: 48


In [60]:
# Save country list
countries_df.to_csv(DATA_DIR / "sub_saharan_countries.csv", index=False)

In [61]:
# Function to pull World Bank indicator data
def fetch_world_bank_indicator(country_codes, indicator_code, column_name):
    codes = ";".join(country_codes)
    
    url = f"https://api.worldbank.org/v2/country/{codes}/indicator/{indicator_code}"
    
    params = {
        "format": "json",
        "per_page": 20000,
        "date": f"{START_YEAR}:{END_YEAR}"
    }
    
    response = requests.get(url, params=params)
    json_data = response.json()
    
    records = json_data[1]
    
    rows = []
    
    for item in records:
        rows.append({
            "country_code": item["countryiso3code"],
            "country_name": item["country"]["value"],
            "year": int(item["date"]),
            column_name: item["value"]
        })
    
    df = pd.DataFrame(rows)
    
    return df


In [63]:
# Get country codes
country_codes = countries_df["country_code"].tolist()

In [64]:
# Pull malaria incidence
malaria_df = fetch_world_bank_indicator(
    country_codes,
    "SH.MLR.INCD.P3",
    "malaria_incidence"
)

# Pull population total
population_df = fetch_world_bank_indicator(
    country_codes,
    "SP.POP.TOTL",
    "population_total"
)

# Pull population density
density_df = fetch_world_bank_indicator(
    country_codes,
    "EN.POP.DNST",
    "population_density"
)

print(malaria_df.head())
print(population_df.head())
print(density_df.head())

  country_code country_name  year  malaria_incidence
0          AGO       Angola  2023             255.47
1          AGO       Angola  2022             260.09
2          AGO       Angola  2021             258.61
3          AGO       Angola  2020             244.53
4          AGO       Angola  2019             243.10
  country_code country_name  year  population_total
0          AGO       Angola  2023          36749906
1          AGO       Angola  2022          35635029
2          AGO       Angola  2021          34532429
3          AGO       Angola  2020          33451132
4          AGO       Angola  2019          32375632
  country_code country_name  year  population_density
0          AGO       Angola  2023           29.477746
1          AGO       Angola  2022           28.583484
2          AGO       Angola  2021           27.699069
3          AGO       Angola  2020           26.831741
4          AGO       Angola  2019           25.969064


##### Merge the World Bank data

In [65]:
# Merge malaria, population, and population density data
wb_df = malaria_df.merge(
    population_df,
    on=["country_code", "country_name", "year"],
    how="left"
)

wb_df = wb_df.merge(
    density_df,
    on=["country_code", "country_name", "year"],
    how="left"
)

print(wb_df.head())
print("Shape:", wb_df.shape)

# Save merged World Bank data
wb_df.to_csv(DATA_DIR / "world_bank_malaria_population_merged.csv", index=False)

  country_code country_name  year  malaria_incidence  population_total  \
0          AGO       Angola  2023             255.47          36749906   
1          AGO       Angola  2022             260.09          35635029   
2          AGO       Angola  2021             258.61          34532429   
3          AGO       Angola  2020             244.53          33451132   
4          AGO       Angola  2019             243.10          32375632   

   population_density  
0           29.477746  
1           28.583484  
2           27.699069  
3           26.831741  
4           25.969064  
Shape: (1152, 6)


##### Pull climate data from NASA POWER

In [81]:
# Function to pull monthly NASA POWER data for one country point
def fetch_nasa_power_monthly(country_code, country_name, latitude, longitude):
    url = "https://power.larc.nasa.gov/api/temporal/monthly/point"
    
    params = {
        "parameters": "T2M,RH2M,PRECTOTCORR",
        "community": "AG",
        "longitude": longitude,
        "latitude": latitude,
        "start": START_YEAR,
        "end": END_YEAR,
        "format": "JSON"
    }
    
    response = requests.get(url, params=params)
    data = response.json()
    
    parameter_data = data["properties"]["parameter"]
    
    rows = []
    
    # NASA monthly keys usually look like YYYYMM, for example 200001 for January 2000
    for date_key in parameter_data["T2M"].keys():
        date_key = str(date_key)
        
        # We only want monthly records like YYYYMM
        if len(date_key) == 6:
            year = int(date_key[:4])
            month = int(date_key[4:])
            
            # Skip strange month values if any appear
            if month < 1 or month > 12:
                continue
            
            days_in_month = calendar.monthrange(year, month)[1]
            
            t2m = parameter_data["T2M"].get(date_key)
            rh2m = parameter_data["RH2M"].get(date_key)
            precip = parameter_data["PRECTOTCORR"].get(date_key)
            
            rows.append({
                "country_code": country_code,
                "country_name": country_name,
                "year": year,
                "month": month,
                "avg_temperature": t2m,
                "avg_relative_humidity": rh2m,
                "precipitation_corrected": precip,
                "days_in_month": days_in_month
            })
    
    monthly_df = pd.DataFrame(rows)
    
    return monthly_df

In [83]:
# Pull climate data for each country
all_climate_monthly = []

for index, row in countries_df.iterrows():
    print("Pulling climate data for:", row["country_name"])
    
    try:
        climate_monthly = fetch_nasa_power_monthly(
            country_code=row["country_code"],
            country_name=row["country_name"],
            latitude=row["latitude"],
            longitude=row["longitude"]
        )
        
        all_climate_monthly.append(climate_monthly)
        
        # Small pause so we do not hit the API too aggressively
        time.sleep(1)
    
    except Exception as e:
        print("Failed for:", row["country_name"])
        print("Error:", e)


# Combine all country climate data
climate_monthly_df = pd.concat(all_climate_monthly, ignore_index=True)

print(climate_monthly_df.head())
print("Shape:", climate_monthly_df.shape)

# Save monthly climate data
climate_monthly_df.to_csv(DATA_DIR / "nasa_power_climate_monthly_raw.csv", index=False)

Pulling climate data for: Angola
Pulling climate data for: Burundi
Pulling climate data for: Benin
Pulling climate data for: Burkina Faso
Pulling climate data for: Botswana
Pulling climate data for: Central African Republic
Pulling climate data for: Cote d'Ivoire
Pulling climate data for: Cameroon
Pulling climate data for: Congo, Dem. Rep.
Pulling climate data for: Congo, Rep.
Pulling climate data for: Comoros
Pulling climate data for: Cabo Verde
Pulling climate data for: Eritrea
Pulling climate data for: Ethiopia
Pulling climate data for: Gabon
Pulling climate data for: Ghana
Pulling climate data for: Guinea
Pulling climate data for: Gambia, The
Pulling climate data for: Guinea-Bissau
Pulling climate data for: Equatorial Guinea
Pulling climate data for: Kenya
Pulling climate data for: Liberia
Pulling climate data for: Lesotho
Pulling climate data for: Madagascar
Pulling climate data for: Mali
Pulling climate data for: Mozambique
Pulling climate data for: Mauritania
Pulling climate dat

In [84]:
# Estimate monthly precipitation amount
climate_monthly_df["estimated_monthly_precipitation"] = (
    climate_monthly_df["precipitation_corrected"] * climate_monthly_df["days_in_month"]
)

# Convert monthly climate records to yearly records
climate_yearly_df = climate_monthly_df.groupby(
    ["country_code", "country_name", "year"],
    as_index=False
).agg({
    "avg_temperature": "mean",
    "avg_relative_humidity": "mean",
    "estimated_monthly_precipitation": "sum"
})

# Rename precipitation column
climate_yearly_df = climate_yearly_df.rename(
    columns={
        "estimated_monthly_precipitation": "estimated_annual_precipitation"
    }
)

print(climate_yearly_df.head())
print("Shape:", climate_yearly_df.shape)

# Save yearly climate data
climate_yearly_df.to_csv(DATA_DIR / "nasa_power_climate_yearly.csv", index=False)

  country_code country_name  year  avg_temperature  avg_relative_humidity  \
0          AGO       Angola  2000        25.340833              78.841667   
1          AGO       Angola  2001        24.808333              80.142500   
2          AGO       Angola  2002        25.309167              79.204167   
3          AGO       Angola  2003        25.955000              77.733333   
4          AGO       Angola  2004        25.353333              78.850833   

   estimated_annual_precipitation  
0                          447.32  
1                          488.06  
2                          304.48  
3                          219.58  
4                          207.84  
Shape: (1152, 6)


In [85]:
# Merge World Bank data with climate data
final_df = wb_df.merge(
    climate_yearly_df,
    on=["country_code", "country_name", "year"],
    how="left"
)

# Add latitude and longitude
final_df = final_df.merge(
    countries_df[["country_code", "capital_city", "latitude", "longitude"]],
    on="country_code",
    how="left"
)

print(final_df.head())
print("Shape:", final_df.shape)

# Check missing values
print(final_df.isnull().sum())

# Save final merged dataset
final_df.to_csv(DATA_DIR / "malaria_climate_population_dataset.csv", index=False)

  country_code country_name  year  malaria_incidence  population_total  \
0          AGO       Angola  2023             255.47          36749906   
1          AGO       Angola  2022             260.09          35635029   
2          AGO       Angola  2021             258.61          34532429   
3          AGO       Angola  2020             244.53          33451132   
4          AGO       Angola  2019             243.10          32375632   

   population_density  avg_temperature  avg_relative_humidity  \
0           29.477746        25.885000              78.878333   
1           28.583484        25.544167              78.264167   
2           27.699069        26.310833              77.334167   
3           26.831741        26.374167              77.599167   
4           25.969064        26.341667              77.260833   

   estimated_annual_precipitation capital_city  latitude  longitude  
0                          375.29       Luanda  -8.81155     13.242  
1                       

##### Create the risk level target

In [86]:
# Drop rows where malaria incidence is missing
model_df = final_df.dropna(subset=["malaria_incidence"]).copy()

# Calculate median malaria incidence
median_malaria = model_df["malaria_incidence"].median()

print("Median malaria incidence:", median_malaria)



Median malaria incidence: 239.66000000000003


In [87]:
print(model_df.isnull().sum())

country_code                       0
country_name                       0
year                               0
malaria_incidence                  0
population_total                   0
population_density                24
avg_temperature                    0
avg_relative_humidity              0
estimated_annual_precipitation     0
capital_city                       0
latitude                           0
longitude                          0
dtype: int64


In [88]:
# Fill missing population density values with the median of that column
density_median = model_df["population_density"].median()
model_df["population_density"] = model_df["population_density"].fillna(density_median)

# Verify they are gone
print("Remaining missing values:", model_df["population_density"].isnull().sum())


Remaining missing values: 0


In [89]:
print(model_df.isnull().sum())

country_code                      0
country_name                      0
year                              0
malaria_incidence                 0
population_total                  0
population_density                0
avg_temperature                   0
avg_relative_humidity             0
estimated_annual_precipitation    0
capital_city                      0
latitude                          0
longitude                         0
dtype: int64


In [90]:
# Create risk level
model_df["risk_level"] = model_df["malaria_incidence"].apply(
    lambda x: 1 if x > median_malaria else 0
)

# Check the target distribution
print(model_df["risk_level"].value_counts())

# Save model-ready dataset
model_df.to_csv(DATA_DIR / "model_ready_malaria_risk_dataset.csv", index=False)

print(model_df.head())
print("Final shape:", model_df.shape)

risk_level
1    540
0    540
Name: count, dtype: int64
  country_code country_name  year  malaria_incidence  population_total  \
0          AGO       Angola  2023             255.47          36749906   
1          AGO       Angola  2022             260.09          35635029   
2          AGO       Angola  2021             258.61          34532429   
3          AGO       Angola  2020             244.53          33451132   
4          AGO       Angola  2019             243.10          32375632   

   population_density  avg_temperature  avg_relative_humidity  \
0           29.477746        25.885000              78.878333   
1           28.583484        25.544167              78.264167   
2           27.699069        26.310833              77.334167   
3           26.831741        26.374167              77.599167   
4           25.969064        26.341667              77.260833   

   estimated_annual_precipitation capital_city  latitude  longitude  \
0                          375.29     

In [94]:
model_df.head()
model_df.shape
model_df.isnull().sum()
model_df["risk_level"].value_counts()

risk_level
1    540
0    540
Name: count, dtype: int64

In [95]:
print(model_df.head())
print("Data Shape:", model_df.shape)
print("\nMissing Values:\n", model_df.isnull().sum())
print("\nTarget Class Distribution:\n", model_df["risk_level"].value_counts())


  country_code country_name  year  malaria_incidence  population_total  \
0          AGO       Angola  2023             255.47          36749906   
1          AGO       Angola  2022             260.09          35635029   
2          AGO       Angola  2021             258.61          34532429   
3          AGO       Angola  2020             244.53          33451132   
4          AGO       Angola  2019             243.10          32375632   

   population_density  avg_temperature  avg_relative_humidity  \
0           29.477746        25.885000              78.878333   
1           28.583484        25.544167              78.264167   
2           27.699069        26.310833              77.334167   
3           26.831741        26.374167              77.599167   
4           25.969064        26.341667              77.260833   

   estimated_annual_precipitation capital_city  latitude  longitude  \
0                          375.29       Luanda  -8.81155     13.242   
1                     

##### Save Full Dataset

In [96]:
# Save the full final dataset for viewing and reference
model_df.to_csv("malaria_climate_population_full_dataset.csv", index=False)

print("Full dataset saved successfully!")

Full dataset saved successfully!


In [97]:
# Create a clean dataset for modelling and viewing
ml_df = model_df[
    [
        "country_name",
        "year",
        "population_total",
        "population_density",
        "avg_temperature",
        "avg_relative_humidity",
        "estimated_annual_precipitation",
        "latitude",
        "longitude",
        "malaria_incidence",
        "risk_level"
    ]
].copy()

# View the first few rows
print(ml_df.head())

# Check the shape
print("Shape:", ml_df.shape)

# Check missing values
print("\nMissing values:\n", ml_df.isnull().sum())

  country_name  year  population_total  population_density  avg_temperature  \
0       Angola  2023          36749906           29.477746        25.885000   
1       Angola  2022          35635029           28.583484        25.544167   
2       Angola  2021          34532429           27.699069        26.310833   
3       Angola  2020          33451132           26.831741        26.374167   
4       Angola  2019          32375632           25.969064        26.341667   

   avg_relative_humidity  estimated_annual_precipitation  latitude  longitude  \
0              78.878333                          375.29  -8.81155     13.242   
1              78.264167                          289.40  -8.81155     13.242   
2              77.334167                          333.94  -8.81155     13.242   
3              77.599167                          298.67  -8.81155     13.242   
4              77.260833                          104.83  -8.81155     13.242   

   malaria_incidence  risk_level  
0  

In [98]:
# Save the clean modelling dataset
ml_df.to_csv("malaria_climate_population_model_dataset.csv", index=False)

print("Clean modelling dataset saved successfully!")

Clean modelling dataset saved successfully!
